# DC-Ops: Fine-tune RetinaNet + Export with QNN HTP for Snapdragon NPU

1. Fine-tune RetinaNet-ResNet50-FPN on 4 Roboflow DC datasets (2,036 images)
2. Download .pth weights immediately
3. Export to ExecuTorch .pte with QNN HTP backend for SM8750

Includes augmentations for moiré patterns, screen glare, and brightness variation.

**Runtime → A100 GPU**

In [ ]:
# 1. Install dependencies + ExecuTorch + QNN SDK
!pip install -q roboflow py-cpuinfo
!git clone --depth 1 https://github.com/pytorch/executorch.git
%cd executorch
!git submodule sync
!git submodule update --init --recursive --depth 1
!bash install_requirements.sh
!pip install -e . --no-build-isolation

# Download QNN SDK
!python backends/qualcomm/scripts/download_qnn_sdk.py --print-sdk-path 2>/dev/null || true
import os
qnn_path = !python backends/qualcomm/scripts/download_qnn_sdk.py --print-sdk-path 2>/dev/null
os.environ['QNN_SDK_ROOT'] = qnn_path[-1].strip() if qnn_path else ''
print(f'QNN SDK: {os.environ.get("QNN_SDK_ROOT", "NOT SET")}')

In [ ]:
# 2. Download Roboflow datasets
import os
os.chdir('/content')

from roboflow import Roboflow
API_KEY = input('Paste your Roboflow API key: ')
rf = Roboflow(api_key=API_KEY)

print('1/4 Ports (ym)...')
rf.workspace('ym-pffnw').project('ports-yutnj').version(1).download('coco', location='roboflow_ports_ym')
print('2/4 Server Detection (Fujitsu)...')
rf.workspace('fujitsu-ih8ei').project('server-detection').version(1).download('coco', location='roboflow_fujitsu')
print('3/4 Server Vision (hunters)...')
rf.workspace('hunters-workspace-kqclz').project('server-vision').version(5).download('coco', location='roboflow_server_vision')
print('4/4 PC Ports...')
rf.workspace('ports-gmmmp').project('pc-ports').version(10).download('coco', location='roboflow_pc_ports')
print('All downloaded!')

In [ ]:
# 3. Merge datasets into COCO format with DC-Ops classes
import json, shutil, glob
from pathlib import Path

DC_CLASSES = [
    'server rack', 'compute tray', 'NVLink switch tray', 'network switch',
    'power shelf', 'cable', 'network port', 'LED indicator',
    'label', 'fan', 'cooling manifold', 'cable cartridge',
    'power connector', 'drive bay', 'management port', 'DPU',
]

PORTS_YM_MAP = {0:6, 1:6, 2:6, 3:12, 4:6, 5:6, 6:6}
FUJITSU_MAP = {0:6, 1:6, 2:6, 3:6, 4:6, 5:15, 6:1, 7:1, 8:6}
SERVER_VISION_MAP = {
    0:1, 1:13, 2:1, 3:1, 4:1, 5:1, 6:13, 7:13, 8:13, 9:9,
    10:13, 11:6, 12:6, 13:10, 14:14, 15:6, 16:14, 17:15, 18:13,
    19:12, 20:12, 21:15, 22:4, 23:15, 24:6, 25:15, 26:14, 27:6, 28:6, 29:6
}
PC_PORTS_MAP = {0:6, 1:6, 2:6, 3:6}

DATASETS = [
    ('roboflow_ports_ym', PORTS_YM_MAP),
    ('roboflow_fujitsu', FUJITSU_MAP),
    ('roboflow_server_vision', SERVER_VISION_MAP),
    ('roboflow_pc_ports', PC_PORTS_MAP),
]

merged_dir = Path('merged_coco')
for split in ['train', 'val']:
    (merged_dir / split).mkdir(parents=True, exist_ok=True)

for dst_split in ['train', 'val']:
    all_images, all_annotations = [], []
    img_id, ann_id = 0, 0
    for ds_path, class_map in DATASETS:
        ds = Path(ds_path)
        for src_split, target_split in [('train', 'train'), ('valid', 'val'), ('test', 'train')]:
            if target_split != dst_split: continue
            ann_file = ds / src_split / '_annotations.coco.json'
            if not ann_file.exists(): continue
            with open(ann_file) as f: coco = json.load(f)
            old_to_new = {}
            for img_info in coco.get('images', []):
                img_id += 1
                old_to_new[img_info['id']] = img_id
                src_img = ds / src_split / img_info['file_name']
                new_name = f"{Path(ds_path).name}_{src_split}_{img_info['file_name']}"
                if src_img.exists(): shutil.copy2(src_img, merged_dir / dst_split / new_name)
                all_images.append({'id': img_id, 'file_name': new_name,
                    'width': img_info.get('width', 640), 'height': img_info.get('height', 640)})
            sorted_ids = sorted(set(c['id'] for c in coco.get('categories', [])))
            src_to_seq = {sid: i for i, sid in enumerate(sorted_ids)}
            for ann in coco.get('annotations', []):
                seq = src_to_seq.get(ann['category_id'])
                if seq is None: continue
                nc = class_map.get(seq)
                if nc is None: continue
                if ann['image_id'] not in old_to_new: continue
                ann_id += 1
                all_annotations.append({'id': ann_id, 'image_id': old_to_new[ann['image_id']],
                    'category_id': nc + 1, 'bbox': ann['bbox'],
                    'area': ann.get('area', ann['bbox'][2]*ann['bbox'][3]), 'iscrowd': 0})
    cats = [{'id': i+1, 'name': n, 'supercategory': 'dc'} for i, n in enumerate(DC_CLASSES)]
    with open(merged_dir / f'annotations_{dst_split}.json', 'w') as f:
        json.dump({'images': all_images, 'annotations': all_annotations, 'categories': cats}, f)
    print(f'{dst_split}: {len(all_images)} images, {len(all_annotations)} annotations')
print('Merged COCO dataset ready!')

In [ ]:
# 4. Fine-tune RetinaNet with moire/screen augmentations
import torch
import torchvision
from torchvision.models.detection import retinanet_resnet50_fpn_v2, RetinaNet_ResNet50_FPN_V2_Weights
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import json, os, time, random

NUM_CLASSES = 16

class ScreenAugment:
    """Simulates moire patterns, screen glare, and brightness artifacts
    that occur when photographing laptop/monitor screens."""
    def __call__(self, img):
        c, h, w = img.shape
        # Moire pattern: overlapping sine waves
        if random.random() < 0.3:
            freq = random.uniform(20, 60)
            angle = random.uniform(0, 3.14)
            y = torch.arange(h).float().unsqueeze(1).expand(h, w)
            x = torch.arange(w).float().unsqueeze(0).expand(h, w)
            pattern = torch.sin(freq * (x * torch.cos(torch.tensor(angle)) + y * torch.sin(torch.tensor(angle))) / w)
            intensity = random.uniform(0.02, 0.08)
            img = img + intensity * pattern.unsqueeze(0)
        # Screen brightness gradient (uneven backlight)
        if random.random() < 0.3:
            gradient = torch.linspace(random.uniform(0.7, 0.9), random.uniform(1.0, 1.3), w)
            img = img * gradient.unsqueeze(0).unsqueeze(0)
        # Color tint (screen white balance shift)
        if random.random() < 0.2:
            tint = torch.tensor([random.uniform(0.95, 1.05) for _ in range(3)]).view(3, 1, 1)
            img = img * tint
        # Scanline effect
        if random.random() < 0.2:
            scanlines = torch.ones(h)
            scanlines[::2] = random.uniform(0.92, 0.98)
            img = img * scanlines.unsqueeze(0).unsqueeze(2)
        return img.clamp(0, 1)

class DCOpsDataset(Dataset):
    def __init__(self, img_dir, ann_file, augment=False):
        with open(ann_file) as f:
            self.coco = json.load(f)
        self.img_dir = img_dir
        self.images = self.coco['images']
        self.img_anns = {}
        for ann in self.coco['annotations']:
            self.img_anns.setdefault(ann['image_id'], []).append(ann)
        base_transforms = [
            transforms.ToTensor(),
            transforms.Resize((640, 640)),
        ]
        if augment:
            base_transforms.extend([
                transforms.RandomApply([transforms.ColorJitter(
                    brightness=0.4, contrast=0.4, saturation=0.3, hue=0.05)], p=0.5),
                transforms.RandomApply([transforms.GaussianBlur(
                    kernel_size=5, sigma=(0.1, 2.0))], p=0.3),
                transforms.RandomAdjustSharpness(sharpness_factor=2.0, p=0.2),
                transforms.RandomAutocontrast(p=0.2),
                ScreenAugment(),
            ])
        self.transform = transforms.Compose(base_transforms)
    def __len__(self): return len(self.images)
    def __getitem__(self, idx):
        info = self.images[idx]
        img = Image.open(os.path.join(self.img_dir, info['file_name'])).convert('RGB')
        ow, oh = img.size
        img = self.transform(img)
        anns = self.img_anns.get(info['id'], [])
        boxes, labels = [], []
        for a in anns:
            x,y,w,h = a['bbox']
            sx, sy = 640.0/ow, 640.0/oh
            x1,y1,x2,y2 = x*sx, y*sy, (x+w)*sx, (y+h)*sy
            if x2>x1 and y2>y1:
                boxes.append([x1,y1,x2,y2])
                labels.append(a['category_id'])
        if not boxes:
            boxes = torch.zeros((0,4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes = torch.tensor(boxes, dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.int64)
        return img, {'boxes': boxes, 'labels': labels}

def collate_fn(batch): return tuple(zip(*batch))

# Load pre-trained RetinaNet, modify head for 16+1 classes
model = retinanet_resnet50_fpn_v2(weights=RetinaNet_ResNet50_FPN_V2_Weights.DEFAULT)
num_anchors = model.head.classification_head.num_anchors
model.head.classification_head.num_classes = NUM_CLASSES + 1
in_channels = model.head.classification_head.cls_logits.in_channels
model.head.classification_head.cls_logits = torch.nn.Conv2d(
    in_channels, num_anchors * (NUM_CLASSES + 1), kernel_size=3, stride=1, padding=1
)

train_ds = DCOpsDataset('merged_coco/train', 'merged_coco/annotations_train.json', augment=True)
val_ds = DCOpsDataset('merged_coco/val', 'merged_coco/annotations_val.json', augment=False)
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, collate_fn=collate_fn, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=4, shuffle=False, collate_fn=collate_fn, num_workers=2)

print(f'Train: {len(train_ds)}, Val: {len(val_ds)}, Classes: {NUM_CLASSES} + background')
print(f'Augmentations: ColorJitter, GaussianBlur, Sharpness, AutoContrast, Moire, Screen glare')

device = torch.device('cuda')
model.to(device)
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

NUM_EPOCHS = 10
start = time.time()
for epoch in range(NUM_EPOCHS):
    model.train()
    epoch_loss = 0
    for images, targets in train_loader:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()
        epoch_loss += losses.item()
    lr_scheduler.step()
    print(f'Epoch {epoch+1}/{NUM_EPOCHS}  loss: {epoch_loss/len(train_loader):.4f}  time: {(time.time()-start)/60:.1f}min')

torch.save(model.state_dict(), '/content/retinanet_dc_ops.pth')
print(f'\nTraining complete!')

In [ ]:
# 4b. DOWNLOAD WEIGHTS IMMEDIATELY (before session dies)
import torch, os

torch.save(model, '/content/retinanet_dc_ops_full.pt')

print(f'retinanet_dc_ops.pth: {os.path.getsize("/content/retinanet_dc_ops.pth") / 1e6:.1f} MB')
print(f'retinanet_dc_ops_full.pt: {os.path.getsize("/content/retinanet_dc_ops_full.pt") / 1e6:.1f} MB')

from google.colab import files
files.download('/content/retinanet_dc_ops.pth')
files.download('/content/retinanet_dc_ops_full.pt')
print('Downloaded! Safe to continue to QNN export.')

In [ ]:
# 5. Load QNN libraries and export
import os, sys, ctypes, glob
os.chdir('/content/executorch')
sys.path.insert(0, '/content/executorch')

qnn_sdk = os.environ.get('QNN_SDK_ROOT', '')
os.environ['LD_LIBRARY_PATH'] = f"{qnn_sdk}/lib/x86_64-linux-clang/:{os.environ.get('LD_LIBRARY_PATH', '')}"
for lib in sorted(glob.glob(f"{qnn_sdk}/lib/x86_64-linux-clang/libQnn*.so")):
    try:
        ctypes.CDLL(lib)
        print(f'  Loaded: {os.path.basename(lib)}')
    except: pass

import torch
from executorch.backends.qualcomm.export_utils import build_executorch_binary, QnnConfig
from executorch.backends.qualcomm.quantizer.quantizer import QuantDtype
from torchvision.models.detection import retinanet_resnet50_fpn_v2

NUM_CLASSES = 16
model = retinanet_resnet50_fpn_v2(weights=None)
num_anchors = model.head.classification_head.num_anchors
model.head.classification_head.num_classes = NUM_CLASSES + 1
in_channels = model.head.classification_head.cls_logits.in_channels
model.head.classification_head.cls_logits = torch.nn.Conv2d(
    in_channels, num_anchors * (NUM_CLASSES + 1), kernel_size=3, stride=1, padding=1
)
model.load_state_dict(torch.load('/content/retinanet_dc_ops.pth', map_location='cpu'))

def forward_without_metrics(self, image):
    features = self.backbone(image)
    return self.head(list(features.values()))
model.forward = lambda img: forward_without_metrics(model, img)
model.eval()

calib_inputs = [(torch.randn(1, 3, 640, 640),) for _ in range(20)]

qnn_config = QnnConfig(
    soc_model="SM8750",
    backend="htp",
    build_folder="/content/qnn_build",
    compile_only=True,
)

print('Building ExecuTorch binary with QNN HTP backend...')
build_executorch_binary(
    model=model,
    qnn_config=qnn_config,
    file_name='/content/dc_ops_retinanet_qnn',
    dataset=calib_inputs,
    quant_dtype=QuantDtype.use_8a8w,
)
print('QNN export complete!')

In [ ]:
# 6. Check and download .pte
import os, glob

ptes = glob.glob('/content/**/*.pte', recursive=True)
print(f'Found .pte files: {ptes}')

for p in ptes:
    if 'retinanet' in p or 'dc_ops' in p:
        print(f'\n{p}: {os.path.getsize(p) / 1e6:.1f} MB')

from google.colab import files
for p in ptes:
    if 'retinanet' in p or 'dc_ops' in p:
        files.download(p)